<a href="https://colab.research.google.com/github/Rohit-U76/RL/blob/main/RL_Assignment_No_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import gymnasium as gym
import numpy as np

def generate_episode(env, policy, max_steps=100):
    """
    Generate an episode following a given policy in the environment.

    Args:
    - env: OpenAI Gym environment
    - policy: array mapping states to actions
    - max_steps: max steps per episode

    Returns:
    - episode: list of (state, action, reward)
    """
    episode = []
    state, _ = env.reset()

    for _ in range(max_steps):
        action = policy[state]
        next_state, reward, terminated, truncated, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state
        if terminated or truncated:
            break
    return episode

def first_visit_mc_prediction(env, policy, num_episodes=1000, gamma=0.99):
    """
    Perform First-Visit Monte Carlo prediction to estimate the value function for a policy.

    Args:
    - env: OpenAI Gym environment
    - policy: array mapping states to actions
    - num_episodes: number of episodes to generate
    - gamma: discount factor

    Returns:
    - V: estimated value function as a numpy array
    """
    num_states = env.observation_space.n
    returns = [[] for _ in range(num_states)]  # store returns for each state
    V = np.zeros(num_states)  # initialize value function

    for _ in range(num_episodes):
        episode = generate_episode(env, policy)
        visited_states = set()

        G = 0  # return
        # Process episode backward
        for t in reversed(range(len(episode))):
            state, _, reward = episode[t]
            G = gamma * G + reward

            # First-visit check
            if state not in visited_states:
                visited_states.add(state)
                returns[state].append(G)
                V[state] = np.mean(returns[state])

    return V

def extract_greedy_policy(env, V, gamma=0.99):
    """
    Extract a greedy policy from the value function.

    Args:
    - env: OpenAI Gym environment
    - V: value function array
    - gamma: discount factor

    Returns:
    - policy: array mapping states to optimal actions
    """
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    policy = np.zeros(num_states, dtype=int)

    for s in range(num_states):
        action_values = np.zeros(num_actions)
        for a in range(num_actions):
            for prob, next_state, reward, done in env.unwrapped.P[s][a]:
                action_values[a] += prob * (reward + gamma * V[next_state])
        policy[s] = np.argmax(action_values)
    return policy

def run_policy(env, policy, max_steps=100):
    """
    Run the environment using the given policy to calculate total rewards.

    Args:
    - env: OpenAI Gym environment
    - policy: array mapping states to actions
    - max_steps: max steps per episode

    Returns:
    - total_reward: cumulative reward obtained
    - path: list of visited states
    """
    state, _ = env.reset()
    total_reward = 0
    path = [state]

    for _ in range(max_steps):
        action = policy[state]
        next_state, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        path.append(next_state)
        state = next_state
        if terminated or truncated:
            break
    return total_reward, path

if __name__ == "__main__":
    # Initialize FrozenLake environment (deterministic)
    env = gym.make("FrozenLake-v1", is_slippery=False)

    # Initialize policy (random initially)
    num_states = env.observation_space.n
    num_actions = env.action_space.n
    policy = np.random.choice(num_actions, size=num_states)

    # Monte Carlo parameters
    num_episodes = 5000
    gamma = 0.99

    # First-Visit Monte Carlo to estimate value function for current policy
    V = first_visit_mc_prediction(env, policy, num_episodes, gamma)

    # Extract greedy policy based on value function
    optimal_policy = extract_greedy_policy(env, V, gamma)

    # Run the optimal policy to find total reward and path
    total_reward, optimal_path = run_policy(env, optimal_policy)

    # Print results
    print("Estimated Value Function (V):")
    print(np.round(V.reshape((4,4)), 3))

    action_symbols = ['←', '↓', '→', '↑']
    print("\nOptimal Policy:")
    print(' '.join(action_symbols[a] for a in optimal_policy.reshape((4,4)).flatten()))
    print(np.array([action_symbols[a] for a in optimal_policy]).reshape(4,4))

    print("\nOptimal Path:")
    print(optimal_path)

    print("\nTotal Reward following optimal policy:", total_reward)

Estimated Value Function (V):
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]

Optimal Policy:
← ← ← ← ← ← ← ← ← ← ← ← ← ← → ←
[['←' '←' '←' '←']
 ['←' '←' '←' '←']
 ['←' '←' '←' '←']
 ['←' '←' '→' '←']]

Optimal Path:
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

Total Reward following optimal policy: 0


Roll No.: C-53
# Assignment 9: First Visit Monte Carlo for FrozenLake

## Objective
Implement the First Visit Monte Carlo prediction method to estimate the value function and derive the optimal policy in the FrozenLake environment.

## Environment
- FrozenLake-v1, 4x4 grid
- Deterministic movement (`is_slippery=False`)

## Method
- Initialize a random policy.
- Generate episodes by following the policy.
- For each episode, calculate returns and update value estimates for first visits of states.
- Extract a greedy policy based on the value function.
- Evaluate the policy by running it in the environment and compute total reward.

## Implementation Details
- **generate_episode:** Simulate an episode following a policy.
- **first_visit_mc_prediction:** Estimate value function with Monte Carlo returns.
- **extract_greedy_policy:** Derive policy that greedily selects actions maximizing expected value.
- **run_policy:** Test the policy and compute cumulative reward.

## Results
- Estimated state-value function for each grid state.
- Optimal policy represented with arrows (← ↓ → ↑).
- The sequence of states traversed by the optimal policy.
- Total reward achieved.

## Conclusion
The First Visit Monte Carlo method effectively estimates the value function and enables derivation of an optimal policy in a model-free manner, demonstrating core principles of reinforcement learning in a tabular environment.